### **<h3 style="color:pink;"> RAG system With LLM Fine-tuning + MLOps Pipeline** 

<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Introduction**</span>

</div>


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Importing libraries**</span>

</div>


In [1]:
import importlib
import datasets
import sentence_transformers

print("✅ No circular import!")

✅ No circular import!


In [2]:
# ✅ Phase 1 - Setup Verification
# Run this cell to confirm all packages are installed correctly

import langchain
import sentence_transformers
import faiss
import pypdf
import pdfplumber
import tiktoken
import pandas as pd
import numpy as np
import ragas
import datasets

print("✅ langchain:", langchain.__version__)
print("✅ sentence_transformers:", sentence_transformers.__version__)
print("✅ faiss: installed")
print("✅ pypdf:", pypdf.__version__)
print("✅ pdfplumber:", pdfplumber.__version__)
print("✅ pandas:", pd.__version__)
print("✅ numpy:", np.__version__)
print("✅ ragas:", ragas.__version__)
print("✅ datasets:", datasets.__version__)
print("")
print("🎉 All packages loaded successfully! You're ready to build!")

✅ langchain: 1.2.10
✅ sentence_transformers: 5.2.3
✅ faiss: installed
✅ pypdf: 6.7.5
✅ pdfplumber: 0.11.9
✅ pandas: 3.0.1
✅ numpy: 2.4.2
✅ ragas: 0.4.3
✅ datasets: 4.6.1

🎉 All packages loaded successfully! You're ready to build!


My environment is 100% ready!

This is what i've built:

RAG_Project/
├── data/
│   ├── raw/          ← PDFs will go here
│   ├── processed/    ← Cleaned text will go here
│   └── embeddings/   ← Vector indexes will go here
├── notebooks/
│   └── 01_data_ingestion.ipynb  ← We are here
├── src/
│   ├── ingestion/
│   ├── retrieval/
│   ├── evaluation/
│   └── serving/
├── models/
├── logs/
└── tests/

<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Data Collection**</span>

</div>


In [3]:
from datasets import load_dataset
import os

os.makedirs("../data/raw", exist_ok=True)

print("⏳ Downloading legal dataset...")

dataset = load_dataset("lighteval/legal_summarization", "BillSum", split="train[:500]")

print(f"✅ Loaded {len(dataset)} legal documents")
print(f"📋 Columns: {dataset.column_names}")
print(f"\n🔍 Sample document preview:")
print(str(dataset[0])[:500])

⏳ Downloading legal dataset...


✅ Loaded 500 legal documents
📋 Columns: ['article', 'summary']

🔍 Sample document preview:
{'article': "SECTION 1. LIABILITY OF BUSINESS ENTITIES PROVIDING USE OF FACILITIES TO NONPROFIT ORGANIZATIONS. (a) Definitions.--In this section: (1) Business entity.--The term ``business entity'' means a firm, corporation, association, partnership, consortium, joint venture, or other form of enterprise. (2) Facility.--The term ``facility'' means any real property, including any building, improvement, or appurtenance. (3) Gross negligence.--The term ``gross negligence'' means voluntary and consc


In [5]:
# Let's explore our dataset properly

print(f"📊 Dataset size: {len(dataset)} documents")
print(f"📋 Columns: {dataset.column_names}")
print("\n" + "="*60)
print("📄 FULL SAMPLE DOCUMENT:")
print("="*60)

sample = dataset[0]
for key, value in sample.items():
    print(f"\n🔑 {key.upper()}:")
    print(str(value)[:300])
    print("-"*40)

📊 Dataset size: 500 documents
📋 Columns: ['article', 'summary']

📄 FULL SAMPLE DOCUMENT:

🔑 ARTICLE:
SECTION 1. LIABILITY OF BUSINESS ENTITIES PROVIDING USE OF FACILITIES TO NONPROFIT ORGANIZATIONS. (a) Definitions.--In this section: (1) Business entity.--The term ``business entity'' means a firm, corporation, association, partnership, consortium, joint venture, or other form of enterprise. (2) Fac
----------------------------------------

🔑 SUMMARY:
Shields a business entity from civil liability relating to any injury or death occurring at a facility of that entity in connection with a use of such facility by a nonprofit organization if: (1) the use occurs outside the scope of business of the business entity; (2) such injury or death occurs dur
----------------------------------------


``article`` → the full legal bill text (this is what we'll index and search)

``summary`` → the human summary (we'll use this later for evaluation)

**Now let's save this data to disk and build the ingestion pipeline.**

In [6]:
import json # a format used to store structured data.
import os # Used to interact with the operating system (files, folders, file size, etc.). things it can do : check file size , create folders , delete files

print("💾 Saving documents to disk...")

documents = []
for i, item in enumerate(dataset):
    doc = {
        "doc_id": f"legal_{i:04d}",
        "text": item["article"],
        "summary": item["summary"],
        "source": "BillSum",
        "doc_type": "legal_bill",
        "index": i
    }
    documents.append(doc)

# Save as JSON
output_path = "../data/raw/legal_documents.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

print(f"✅ Saved {len(documents)} documents to {output_path}")
print(f"📁 File size: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB")
print(f"\n🔍 Sample saved document:")
print(json.dumps(documents[0], indent=2)[:400])

💾 Saving documents to disk...
✅ Saved 500 documents to ../data/raw/legal_documents.json
📁 File size: 4.77 MB

🔍 Sample saved document:
{
  "doc_id": "legal_0000",
  "text": "SECTION 1. LIABILITY OF BUSINESS ENTITIES PROVIDING USE OF FACILITIES TO NONPROFIT ORGANIZATIONS. (a) Definitions.--In this section: (1) Business entity.--The term ``business entity'' means a firm, corporation, association, partnership, consortium, joint venture, or other form of enterprise. (2) Facility.--The term ``facility'' means any real property, includ


🎉 4.77MB of real legal documents saved to disk!

**Now the most important part of Phase 1 — chunking. This is where we split long documents into smaller pieces that can be searched efficiently.**

#### Step 1 : Chunking (no model needed!): only chunk size and chunk overlap

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json

# Load our saved documents
with open("../data/raw/legal_documents.json", "r", encoding="utf-8") as f:
    documents = json.load(f)

print(f"📂 Loaded {len(documents)} documents")

# ── Chunking ──────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,        # max characters per chunk
    chunk_overlap=50,      # overlap between chunks to preserve context
    separators=["\n\n", "\n", ". ", " ", ""]  # split priority
)

chunks = []
for doc in documents:
    doc_chunks = splitter.split_text(doc["text"])
    for i, chunk_text in enumerate(doc_chunks):
        chunks.append({
            "chunk_id": f"{doc['doc_id']}_chunk_{i:03d}",
            "doc_id": doc["doc_id"],
            "text": chunk_text,
            "chunk_index": i,
            "total_chunks": len(doc_chunks),
            "source": doc["source"],
            "doc_type": doc["doc_type"]
        })

print(f"✅ Created {len(chunks)} chunks from {len(documents)} documents")
print(f"📊 Average chunks per document: {len(chunks)/len(documents):.1f}")
print(f"\n🔍 Sample chunk:")
print(json.dumps(chunks[0], indent=2))

📂 Loaded 500 documents
✅ Created 11954 chunks from 500 documents
📊 Average chunks per document: 23.9

🔍 Sample chunk:
{
  "chunk_id": "legal_0000_chunk_000",
  "doc_id": "legal_0000",
  "text": "SECTION 1. LIABILITY OF BUSINESS ENTITIES PROVIDING USE OF FACILITIES TO NONPROFIT ORGANIZATIONS. (a) Definitions.--In this section: (1) Business entity.--The term ``business entity'' means a firm, corporation, association, partnership, consortium, joint venture, or other form of enterprise. (2) Facility.--The term ``facility'' means any real property, including any building, improvement, or appurtenance",
  "chunk_index": 0,
  "total_chunks": 13,
  "source": "BillSum",
  "doc_type": "legal_bill"
}


🎉 11,954 chunks created!

1 document ÷ 512 characters = ~23 chunks per document

500 documents × 23 chunks = ~11,500 chunks
                           ≈ 11,954 chunks (exact number)

**Let's save the chunks to disk then build the embeddings.**

In [8]:
# Save chunks to disk
output_path = "../data/processed/chunks.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f"✅ Saved {len(chunks)} chunks to {output_path}")
print(f"📁 File size: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB")

# Quick statistics
chunk_lengths = [len(c["text"]) for c in chunks]
print(f"\n📊 Chunk Statistics:")
print(f"   Min length  : {min(chunk_lengths)} characters")
print(f"   Max length  : {max(chunk_lengths)} characters")
print(f"   Avg length  : {sum(chunk_lengths)/len(chunk_lengths):.0f} characters")

✅ Saved 11954 chunks to ../data/processed/chunks.json
📁 File size: 6.54 MB

📊 Chunk Statistics:
   Min length  : 7 characters
   Max length  : 512 characters
   Avg length  : 371 characters


🎉 Perfect! Chunks saved and statistics look healthy.

Now the exciting part: creating embeddings. 

This converts each text chunk into a vector (list of numbers) that captures its meaning, so we can search by similarity later.

#### Step 2 : **Embedding (Model needed!)** 
``model = SentenceTransformer("all-MiniLM-L6-v2")`` ,
``embedding = model.encode(texts)``

In [9]:
from sentence_transformers import SentenceTransformer
import numpy as np
import time

# Load the embedding model (downloads ~90MB first time)
print("⏳ Loading embedding model... (downloads ~90MB on first run)")
model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Model loaded!")

# Extract just the text from chunks
texts = [chunk["text"] for chunk in chunks]

print(f"\n⏳ Embedding {len(texts)} chunks... (this takes 3-8 minutes on CPU)")
print("☕ Good time for a coffee break!\n")

start = time.time()

# Embed in batches of 64 for memory efficiency
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

elapsed = time.time() - start
print(f"\n✅ Done! Embedded {len(embeddings)} chunks in {elapsed:.0f} seconds")

⏳ Loading embedding model... (downloads ~90MB on first run)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded!

⏳ Embedding 11954 chunks... (this takes 3-8 minutes on CPU)
☕ Good time for a coffee break!



Batches:   0%|          | 0/187 [00:00<?, ?it/s]


✅ Done! Embedded 11954 chunks in 307 seconds


In [10]:
print(f"📐 Embedding shape: {embeddings.shape}")

📐 Embedding shape: (11954, 384)


In [11]:
print(f"📐 Each chunk → vector of {embeddings.shape[1]} numbers")

📐 Each chunk → vector of 384 numbers


In [12]:
import faiss
import os

print("🔨 Building FAISS index...")

dimension = embeddings.shape[1]  # 384
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype(np.float32))

print(f"✅ FAISS index built!")
print(f"📊 Total vectors indexed: {index.ntotal}")

🔨 Building FAISS index...
✅ FAISS index built!
📊 Total vectors indexed: 11954


In [13]:
# Save index to disk
os.makedirs("../data/embeddings", exist_ok=True)
faiss.write_index(index, "../data/embeddings/faiss_index.bin")

# Save embeddings as numpy file
np.save("../data/embeddings/embeddings.npy", embeddings)

print(f"💾 Index saved!")
print(f"📁 Index size: {os.path.getsize('../data/embeddings/faiss_index.bin') / 1024 / 1024:.2f} MB")

💾 Index saved!
📁 Index size: 17.51 MB


🎉 My search engine is built 

17.51MB index holding 11,954 legal document vectors — ready for instant similarity search!

Let's test it with a real query. 

In [14]:
# 🔍 TEST: Search the index with a real legal question

def search(query, top_k=5):
    # Embed the query
    query_vector = model.encode([query], convert_to_numpy=True).astype(np.float32)
    
    # Search FAISS (Facebook AI Similarity Search)
    distances, indices = index.search(query_vector, top_k)
    
    # Return matching chunks
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        results.append({
            "chunk_id": chunks[idx]["chunk_id"],
            "doc_id": chunks[idx]["doc_id"],
            "text": chunks[idx]["text"],
            "distance": float(dist)
        })
    return results

# Test with a real legal query
query = "What are the liability rules for business entities?"
results = search(query, top_k=3)

print(f"🔍 Query: '{query}'")

🔍 Query: 'What are the liability rules for business entities?'


In [15]:
print(f"{'='*60}")
for i, r in enumerate(results):
    print(f"\n📄 Result {i+1} | {r['doc_id']} | distance: {r['distance']:.4f}")
    print(f"{r['text'][:300]}")
    print("-"*60)


📄 Result 1 | legal_0000 | distance: 0.5683
. (b) Limitation on Liability.-- (1) In general.--Subject to subsection (c), a business entity shall not be subject to civil liability relating to any injury or death occurring at a facility of the business entity in connection with a use of such facility by a nonprofit organization if-- (A) the use
------------------------------------------------------------

📄 Result 2 | legal_0000 | distance: 0.7418
SECTION 1. LIABILITY OF BUSINESS ENTITIES PROVIDING USE OF FACILITIES TO NONPROFIT ORGANIZATIONS. (a) Definitions.--In this section: (1) Business entity.--The term ``business entity'' means a firm, corporation, association, partnership, consortium, joint venture, or other form of enterprise. (2) Fac
------------------------------------------------------------

📄 Result 3 | legal_0171 | distance: 0.8590
. (4) The protection from liability shall apply only if the organization or entity provides a financially secure source of recovery for individu

User: "What are liability rules?"
         ↓
    MiniLM converts to vector
         ↓
    FAISS finds top 3 similar chunks
         ↓
    Llama reads those 3 chunks
         ↓
User gets answer: "Business entities are not 
liable when use occurs outside business scope

🎉 MY RAG SEARCH ENGINE IS WORKING PERFECTLY!
Look at what just happened:

I asked a natural language legal question

It found the most relevant chunks in under a second

Result 1 is about liability limitations — exactly what you asked!

Distance 0.57 = very close match, 0.86 = less relevant — the ranking is correct!

**Summary for all what i did till now!**

In [16]:
import os

print("=" * 60)
print("🎉 PHASE 1 - DAY 1 & 2 COMPLETE!")
print("=" * 60)

print("\n📦 What we built today:")
print("   ✅ Project structure created")
print("   ✅ Virtual environment configured")
print("   ✅ All packages installed")
print("   ✅ 500 real legal documents downloaded")
print("   ✅ 11,954 chunks created with metadata")
print("   ✅ Embeddings generated (384 dimensions)")
print("   ✅ FAISS search index built & saved")
print("   ✅ Semantic search working!")

print("\n📁 Files saved to disk:")
for path in [
    "../data/raw/legal_documents.json",
    "../data/processed/chunks.json",
    "../data/embeddings/faiss_index.bin",
    "../data/embeddings/embeddings.npy"
]:
    size = os.path.getsize(path) / 1024 / 1024
    print(f"   📄 {path.split('/')[-1]:<30} {size:.2f} MB")

print("\n🔜 Next session - Week 2:")
print("   → Build RAGAS evaluation framework")
print("   → Create 200 question-answer pairs")
print("   → Measure baseline retrieval quality")
print("   → Track scores in MLflow")
print("\n" + "=" * 60)

🎉 PHASE 1 - DAY 1 & 2 COMPLETE!

📦 What we built today:
   ✅ Project structure created
   ✅ Virtual environment configured
   ✅ All packages installed
   ✅ 500 real legal documents downloaded
   ✅ 11,954 chunks created with metadata
   ✅ Embeddings generated (384 dimensions)
   ✅ FAISS search index built & saved
   ✅ Semantic search working!

📁 Files saved to disk:
   📄 legal_documents.json           4.77 MB
   📄 chunks.json                    6.54 MB
   📄 faiss_index.bin                17.51 MB
   📄 embeddings.npy                 17.51 MB

🔜 Next session - Week 2:
   → Build RAGAS evaluation framework
   → Create 200 question-answer pairs
   → Measure baseline retrieval quality
   → Track scores in MLflow



RAG_Project/
├── data/
│   ├── raw/
│   │   └── legal_documents.json    ✅ 4.77 MB
│   ├── processed/
│   │   └── chunks.json             ✅ 6.54 MB
│   └── embeddings/
│       ├── faiss_index.bin         ✅ 17.51 MB
│       └── embeddings.npy          ✅ 17.51 MB
├── notebooks/
│   └── 01_data_ingestion.ipynb     ✅ Done

🔜 Next Session — Week 2: Evaluation Framework

I'll build:

200 question-answer pairs from your legal documents:  MY ground truth **benchmark**

**RAGAS evaluation pipeline :**
- -  measures faithfulness, 
- - relevancy, 
- - context precision

**Baseline scores :**  documented so every future improvement is measurable

**MLflow tracking :** logs every experiment automatically

_____ 

🎯 Week 3 Goal : Beat our baseline scores by trying:

- Different chunk sizes (256, 512, 1024)
- Better embedding models (BGE-large vs MiniLM)
- Log everything to MLflow and compare!

For week 3 : first thing is to activate the venv on powershell : 
``cd C:\Users\USER\Documents\RAG_Project``
``venv\Scripts\activate``